In [27]:
%load_ext autoreload
%autoreload 2

import os
import plots
import utils as u
import numpy as np
import matplotlib.pyplot as plt

colors = plots.COLORS
plot_path = "./figures/ExampleEKF/"
os.makedirs(plot_path, exist_ok=True)
C = 299792.458 # km/s

flag_save = False
flag_show = True
save_kwargs = {"dpi": 300, "bbox_inches": "tight"}
%matplotlib qt5

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
data = u.load_data("ExampleEKF")
t = data["rv"][:,0] / 3600
n = int(len(t))
t = t[:n]

In [29]:
plt.figure()
plt.step(t, np.sum(data["CN0"][:,1:] > 15, axis=1))
plt.xlabel("Time [h]")
plt.ylabel("Number of satellites")
plt.grid()

In [30]:
# CN0
plt.figure()
for i in range(data["CN0"].shape[1]-1):
    plt.scatter(t, data["CN0"][:n,i+1], s=0.5)

2023-09-12 19:29:31.414 python[51164:2732946] _TIPropertyValueIsValid called with 4 on nil context!
2023-09-12 19:29:31.414 python[51164:2732946] imkxpc_getApplicationProperty:reply: called with incorrect property value 4, bailing.
2023-09-12 19:29:31.414 python[51164:2732946] Text input context does not respond to _valueForTIProperty:
2023-09-12 19:29:31.414 python[51164:2732946] _TIPropertyValueIsValid called with 4 on nil context!
2023-09-12 19:29:31.414 python[51164:2732946] imkxpc_getApplicationProperty:reply: called with incorrect property value 4, bailing.
2023-09-12 19:29:31.414 python[51164:2732946] Text input context does not respond to _valueForTIProperty:


In [31]:
# Plot moon
fig = plots.Plot3D(elev=25, azim=-50)
fig.plot_surface("MOON")

keys = ["rv", "rv_est"]
labels = ["True", "Estimate"]
for key, label in zip(keys, labels):
    fig.plot(data[key][:,1:4], label=label)
fig.label_axis()
plt.title("Satellite orbit (MI)")
plt.legend()

if flag_save:
    plt.savefig(os.path.join(plot_path, "1_MoonSatMI.png"), **save_kwargs)

if flag_show:
    plt.show()
else:
    plt.close()

In [32]:
# 8 plots for rx, ry, rz, vx, vy, vz, bias, drift
nsigma = 3
ax_labels = ["X", "Y", "Z"]
plt.figure(figsize=(10,8))
for i in range(3):
    xyz = ax_labels[i]

    # Position
    plt.subplot(4,2,i*2+1)
    plt.plot(t, data["rv"][:n,i+1], "--", lw=3, label="True")
    plt.plot(t, data["rv_est"][:n,i+1], label="Estimate")
    plt.plot(t, data["rv_pred_only"][:n,i+1], label="No EKF")
    plt.ylabel(f"$r_{xyz}$ [km]")
    plt.grid()

    # Velocity
    plt.subplot(4,2,i*2+2)
    plt.plot(t, data["rv"][:n,i+4]*1e3, "--", lw=3, label="True")
    plt.plot(t, data["rv_est"][:n,i+4]*1e3, label="Estimate")
    plt.plot(t, data["rv_pred_only"][:n,i+4]*1e3, label="No EKF")
    plt.ylabel(f"$v_{xyz}$ [m/s]")
    plt.grid()

# Bias
plt.subplot(4,2,7)
plt.plot(t, data["clk"][:n,1] * C, "--", lw=3, label="True")
plt.plot(t, data["clk_est"][:n,1] * C, label="Estimate")
# plt.plot(t, data["clk_pred_only"][:n,1] * C, label="No EKF")
plt.ylabel("$b$ [km]")
plt.grid()

plt.legend()
plt.xlabel("Time [h]")

# Drift
plt.subplot(4,2,8)
plt.plot(t, data["clk"][:n,2] * C * 1e3, "--", lw=3, label="True")
plt.plot(t, data["clk_est"][:n,2] * C * 1e3, label="Estimate")
# plt.plot(t, data["clk_pred_only"][:n,2] * C * 1e3, label="No EKF")
plt.ylabel("$d$ [m/s]")
plt.grid()

plt.legend()
plt.xlabel("Time [h]")

plt.suptitle("Position and Velocity Estimates")
plt.tight_layout()

if flag_save:
    plt.savefig(os.path.join(plot_path, "2_PositionVelocity.png"), **save_kwargs)

if flag_show:
    plt.show()
else:
    plt.close()

In [33]:
# 8 plots for error in  rx, ry, rz, vx, vy, vz, bias, drift (EKF)
nsigma = 3
ax_labels = ["X", "Y", "Z"]

flag_set_ylim = True
if flag_set_ylim:
    ylim_pos = 1000 * np.array([-1, 1])
    ylim_vel = 300 * np.array([-1, 1])
else:
    ylim_pos = None
    ylim_vel = None

plt.figure(figsize=(10, 8))
for i in range(3):
    xyz = ax_labels[i]
    # Position
    plt.subplot(4, 2, i * 2 + 1)
    plt.plot(t, (data["rv_est"][:n, i + 1] - data["rv"][:n, i + 1]) * 1e3, label="True")
    plt.fill_between(
        t,
        -nsigma * data["P_rv"][:n, i + 1] ** 0.5 * 1e3,
        +nsigma * data["P_rv"][:n, i + 1] ** 0.5 * 1e3,
        alpha=0.2,
    )
    plt.ylabel(f"$\hat{{r}}_{xyz} - r_{xyz}$ [m]")
    plt.grid()
    if ylim_pos is not None:
        plt.ylim(ylim_pos)

    # Velocity
    plt.subplot(4, 2, i * 2 + 2)
    plt.plot(t, (data["rv_est"][:n, i + 4] - data["rv"][:n, i + 4]) * 1e6, label="True")
    plt.fill_between(
        t,
        -nsigma * data["P_rv"][:n, i + 4] ** 0.5 * 1e6,
        +nsigma * data["P_rv"][:n, i + 4] ** 0.5 * 1e6,
        alpha=0.2,
    )
    plt.ylabel(f"$\hat{{v}}_{xyz} - v_{xyz}$ [mm/s]")
    plt.grid()
    if ylim_vel is not None:
        plt.ylim(ylim_vel)

# Bias
plt.subplot(4, 2, 7)
plt.plot(t, (data["clk_est"][:n, 1] - data["clk"][:n, 1]) * C * 1e3, label="True")
plt.fill_between(
    t,
    -nsigma * data["P_clk"][:n, 1] ** 0.5 * C * 1e3,
    +nsigma * data["P_clk"][:n, 1] ** 0.5 * C * 1e3,
    alpha=0.2,
)
plt.ylabel("$\hat{b} - b$ [m]")
plt.grid()
if ylim_pos is not None:
    plt.ylim(ylim_pos)

# Drift
plt.subplot(4, 2, 8)
plt.plot(t, (data["clk_est"][:n, 2] - data["clk"][:n, 2]) * C * 1e6, label="True")
plt.fill_between(
    t,
    -nsigma * data["P_clk"][:n, 2] ** 0.5 * C * 1e6,
    +nsigma * data["P_clk"][:n, 2] ** 0.5 * C * 1e6,
    alpha=0.2,
)
plt.ylabel("$\hat{d} - d$ [mm/s]")
plt.grid()
if ylim_vel is not None:
    plt.ylim(ylim_vel)

# Labels
plt.subplot(4, 2, 7)
plt.xlabel("Time [h]")
plt.subplot(4, 2, 8)
plt.xlabel("Time [h]")

# Title
plt.suptitle("EKF Position and Velocity Error ($3\sigma$)")
plt.tight_layout()

# Save and show
if flag_save:
    plt.savefig(
        os.path.join(plot_path, "3_PositionVelocityErrorEKF.png"), **save_kwargs
    )
if flag_show:
    plt.show()
else:
    plt.close()

In [34]:
# 6 plots for error in  rx, ry, rz, vx, vy, vz (NO EKF)

nsigma = 3
ax_labels = ["X", "Y", "Z"]
plt.figure(figsize=(10,8))

for i in range(3):
    xyz = ax_labels[i]
    plt.subplot(3,2,i*2+1)
    plt.plot(t, 
             data["rv_pred_only"][:n,i+1] - data["rv"][ :n,i+1], label="True")
    plt.fill_between(t, 
                     -nsigma*data["P_rv_pred_only"][:n,i+1]**0.5, 
                     +nsigma*data["P_rv_pred_only"][:n,i+1]**0.5, alpha=0.2)
    plt.ylabel(f"$\hat{{r}}_{xyz} - r_{xyz}$ [km]")
    plt.grid()

    plt.subplot(3,2,i*2+2)
    plt.plot(t, 
             (data["rv_pred_only"][:n,i+4] - data["rv"][:n,i+4])*1e3, label="True")
    plt.fill_between(t, 
                     -nsigma*data["P_rv_pred_only"][:n,i+4]**0.5*1e3, 
                     +nsigma*data["P_rv_pred_only"][:n,i+4]**0.5*1e3, alpha=0.2)
    plt.ylabel(f"$\hat{{v}}_{xyz} - v_{xyz}$ [m/s]")
    plt.grid()

plt.subplot(3,2,5)
plt.xlabel("Time [h]")

plt.subplot(3,2,6)
plt.xlabel("Time [h]")

plt.suptitle("Position and Velocity Error (No EKF)")
plt.tight_layout()

if flag_save:
    plt.savefig(os.path.join(plot_path, "4_PositionVelocityErrorEKFOff.png"), **save_kwargs)

if flag_show:
    plt.show()
else:
    plt.close()

In [35]:
# 2 plots for error in r and v magnitude (EKF)
r_err = np.linalg.norm(data["rv_est"][:,1:4] - data["rv"][:,1:4], axis=1)
v_err = np.linalg.norm(data["rv_est"][:,4:7] - data["rv"][:,4:7], axis=1)
r_std = np.sqrt(np.linalg.norm(data["P_rv"][:,1:4], axis=1))
v_std = np.sqrt(np.linalg.norm(data["P_rv"][:,4:7], axis=1))

plt.figure(figsize=(6,5))
plt.subplot(2,1,1)
plt.plot(t, r_err[:n])
plt.fill_between(t, -nsigma*r_std[:n], +nsigma*r_std[:n], alpha=0.2)
plt.ylabel("Position error [km]")
plt.xlabel("Time [h]")
plt.grid()

plt.subplot(2,1,2)
plt.plot(t, v_err[:n]*1e3)
plt.fill_between(t, -nsigma*v_std[:n]*1e3, +nsigma*v_std[:n]*1e3, alpha=0.2)
plt.ylabel("Velocity error [m/s]")
plt.xlabel("Time [h]")
plt.grid()

plt.suptitle("Position and Velocity Error (No EKF)")
plt.tight_layout()

if flag_save:
    plt.savefig(os.path.join(plot_path, "5_PositionVelocityErrorNormEKF.png"), **save_kwargs)

if flag_show:
    plt.show()
else:
    plt.close()

In [36]:
# 2 plots for error in r and v magnitude (NO EKF)

r_err_pred_only = np.linalg.norm(data["rv_pred_only"][:,1:4] - data["rv"][:,1:4], axis=1)
v_err_pred_only = np.linalg.norm(data["rv_pred_only"][:,4:7] - data["rv"][:,4:7], axis=1)
r_std_pred_only = np.sqrt(np.linalg.norm(data["P_rv_pred_only"][:,1:4], axis=1))
v_std_pred_only = np.sqrt(np.linalg.norm(data["P_rv_pred_only"][:,4:7], axis=1))

plt.figure(figsize=(6,5))
plt.subplot(2,1,1)
plt.plot(t, r_err_pred_only[:n])
plt.fill_between(t, -nsigma*r_std_pred_only[:n], +nsigma*r_std_pred_only[:n], alpha=0.2)
plt.ylabel("Position error [km]")
plt.xlabel("Time [h]")
plt.grid()

plt.subplot(2,1,2)
plt.plot(t, v_err_pred_only[:n]*1e3)
plt.fill_between(t, -nsigma*v_std_pred_only[:n]*1e3, +nsigma*v_std_pred_only[:n]*1e3, alpha=0.2)
plt.ylabel("Velocity error [m/s]")
plt.xlabel("Time [h]")
plt.grid()

plt.suptitle("Position and Velocity Error (No EKF)")
plt.tight_layout()

if flag_save:
    plt.savefig(os.path.join(plot_path, "6_PositionVelocityErrorNormEKFOff.png"), **save_kwargs)

if flag_show:
    plt.show()

In [37]:
# 2 plots for error in clk bias and drift

nsigma = 3


plt.figure(figsize=(6,5))
plt.subplot(2,1,1)
plt.plot(t, (data["clk_est"][:n, 1] - data["clk"][:n, 1]) * C)
plt.fill_between(t, 
                 -nsigma*data["P_clk"][:n,1] * C, 
                 +nsigma*data["P_clk"][:n,1] * C, alpha=0.2)
plt.ylabel("Clock bias [km]")
plt.xlabel("Time [h]")
plt.grid()

plt.subplot(2,1,2)
plt.plot(t, (data["clk_est"][:n, 2] - data["clk"][:n, 2]) * C * 1e3)
plt.fill_between(t, 
                 -nsigma*data["P_clk"][:n,2] * C * 1e3, 
                 +nsigma*data["P_clk"][:n,2] * C * 1e3, alpha=0.2)
plt.ylabel("Clock drift [m/s]")
plt.xlabel("Time [h]")
plt.grid()

plt.title("Clock Bias and Drift Error")
plt.tight_layout()

if flag_save:
    plt.savefig(os.path.join(plot_path, "7_ClockDriftBias.png"), **save_kwargs)

if flag_show:
    plt.show()

2023-09-12 19:29:37.699 python[51164:2732946] IMKClient Stall detected, *please Report* your user scenario attaching a spindump (or sysdiagnose) that captures the problem - (imkxpc_bundleIdentifierWithReply:) block performed very slowly (1.09 secs).
2023-09-12 19:29:37.699 python[51164:2732946] IMKClient Stall detected, *please Report* your user scenario attaching a spindump (or sysdiagnose) that captures the problem - (imkxpc_dismissFunctionRowItemTextInputViewWithReply:) block performed very slowly (1.09 secs).


In [38]:
# CN0 and number of satellites in view

plt.figure(figsize=(6,5))
plt.subplot(2,1,1)
t = data["CN0"][:,0] / 3600
cn0 = data["CN0"][:,1:]
for i in range(cn0.shape[1]):
    plt.scatter(t, cn0[:,i], s=0.2)
plt.xlabel("Time [h]")
plt.ylabel("C/N0 [dB-Hz]")
plt.title("Carrier-to-noise density ratio")
plt.grid()

plt.subplot(2,1,2)
plt.step(t, np.sum(~np.isnan(cn0), axis=1))
plt.xlabel("Time [h]")
plt.ylabel("Number of satellites")
plt.title("Number of satellites in view")
plt.grid()

plt.tight_layout()

if flag_save:
    plt.savefig(os.path.join(plot_path, "8_CNOSatelliteCount.png"), **save_kwargs)

if flag_show:
    plt.show()

In [39]:
# Pseudorange measurement

plt.figure(figsize=(10, 5))
plt.plot(t, (data["z_pr_pred"][:,1:] - data["z_pr"][:,1:])*1e3, ":", label="Estimated")
plt.title("Pseudorange Residuals")
plt.xlabel("Time [h]")
plt.ylabel("Residual [m]")
plt.grid()
plt.title("Pseudorange Residuals")

if flag_save:
    plt.savefig(os.path.join(plot_path, "9_PseudorangeResiduals.png"), **save_kwargs)

if flag_show:
    plt.show()

2023-09-12 19:30:44.325 python[51164:2732946] _TIPropertyValueIsValid called with 4 on nil context!
2023-09-12 19:30:44.325 python[51164:2732946] imkxpc_getApplicationProperty:reply: called with incorrect property value 4, bailing.
2023-09-12 19:30:44.325 python[51164:2732946] Text input context does not respond to _valueForTIProperty:
2023-09-12 19:30:44.328 python[51164:2732946] _TIPropertyValueIsValid called with 4 on nil context!
2023-09-12 19:30:44.328 python[51164:2732946] imkxpc_getApplicationProperty:reply: called with incorrect property value 4, bailing.
2023-09-12 19:30:44.328 python[51164:2732946] Text input context does not respond to _valueForTIProperty:
